# Python Modules, Packages & Imports — Complete Notes

A single reference notebook covering everything discussed: modules, packages,
`__init__.py`, import styles, `__name__`/`__main__`, `sys.path`, `sys.modules`,
namespace packages, dunders, and `unittest` — with runnable examples throughout.

**How to use this notebook:** run cells top to bottom. Some cells write temporary
files/folders to a `nb_workspace/` folder (cleaned up at the start) so that import
examples are genuinely runnable, not just illustrative.

---
## Table of Contents
1. [Setup](#1)
2. [Modules — the basics](#2)
3. [Packages and `__init__.py`](#3)
4. [What `__init__.py` can customize/do](#4)
5. [Import styles: absolute vs relative](#5)
6. [Why relative imports need `from ... import ...`](#6)
7. [`__name__` and the `__main__` guard](#7)
8. [Running a package: `__main__.py` and `python -m`](#8)
9. [`__main__.py` for an ML engineer](#9)
10. [`PATH` vs `sys.path`](#10)
11. [Fixing import errors by inserting a path](#11)
12. [`sys.modules` and the import/memory lifecycle](#12)
13. [Namespace packages (no `__init__.py`)](#13)
14. [Inspecting your namespace cleanly](#14)
15. [Dunders reference table](#15)
16. [`unittest` — how it works](#16)
17. [Summary cheat-sheet](#17)


<a id='1'></a>
## 1. Setup

Clean workspace used for every runnable example below.

In [2]:
import os, sys, shutil
from pathlib import Path

WORKDIR = Path.cwd() / "nb_workspace"
if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
WORKDIR.mkdir()
os.chdir(WORKDIR)
print("Working in:", Path.cwd())


Working in: c:\Users\ACER\OneDrive\Documents\Repositories\my_learning\AI_ML\programming_math\python\nb_workspace


<a id='2'></a>
## 2. Modules — the basics

A **module** is just a `.py` file. Its contents (functions, classes, variables)
become attributes of a module object once imported.

In [3]:
Path("math_utils.py").write_text('''
def add(a, b):
    return a + b

PI = 3.14159
''')

import math_utils
print(math_utils.add(2, 3))
print(math_utils.PI)


5
3.14159


<a id='3'></a>
## 3. Packages and `__init__.py`

A **package** is a directory of modules. Traditionally it needs an `__init__.py`
file to be recognized as a package (Python 3.3+ also allows "namespace packages"
without one — see [section 13](#13)).

In [4]:
pkg = Path("mypackage")
pkg.mkdir()
(pkg / "__init__.py").write_text("")  # empty for now
(pkg / "data_utils.py").write_text('''
def clean_data(x):
    return x.strip().lower()
''')

import mypackage.data_utils
print(mypackage.data_utils.clean_data("  HELLO  "))


hello


<a id='4'></a>
## 4. What `__init__.py` can customize/do

`__init__.py` runs **once**, the first time anything in the package is imported
(including importing just one submodule — the parent has to load first).
It gives you three real capabilities:

1. **Package-level initialization code** — set up version info, config, logging defaults.
2. **Control what `from package import *` exposes**, via `__all__`.
3. **Re-export names from submodules** for a shorter public API.

Example below rebuilds `mypackage` to demonstrate all three at once.

In [21]:
shutil.rmtree(pkg)
pkg.mkdir()

(pkg / "data_utils.py").write_text('''
def clean_data(x):
    return x.strip().lower()

def _internal_helper():
    return "not meant for public use"
''')

(pkg / "model_utils.py").write_text('''
def train_model(data):
    return f"trained on {len(data)} rows"
''')

(pkg / "__init__.py").write_text('''
# mypackage: demonstrates __init__.py capabilities.
import logging

# 1. Package-level initialization
__version__ = "1.0.0"
logging.getLogger(__name__).addHandler(logging.NullHandler())
print("mypackage is initializing...")

# 3. Re-export from submodules (short import path for users)
from .data_utils import clean_data
from .model_utils import train_model

# 2. Control star-imports
__all__ = ["clean_data", "train_model"]
''')

# Fresh interpreter view: clear any cached copies from the cell above
for name in list(sys.modules):
    if name == "mypackage" or name.startswith("mypackage."):
        del sys.modules[name]

import mypackage
print("version:", mypackage.__version__)
print("short path works:", mypackage.clean_data("  HI  "))

mypackage is initializing...
version: 1.0.0
short path works: hi


In [22]:
# __all__ in action: only clean_data and train_model survive a star-import
ns = {}
exec("from mypackage import *", ns)
exposed = sorted(k for k in ns if not k.startswith("__"))
print("exposed by import *:", exposed)
print("_internal_helper reachable via *?", "_internal_helper" in ns)
print("logging reachable via *?", "logging" in ns)  # excluded too, despite no leading _,
                                                    # because __all__ restricts it explicitly


exposed by import *: ['clean_data', 'train_model']
_internal_helper reachable via *? False
logging reachable via *? False


**Without `__all__`**, a star-import instead grabs every name in `__init__.py`'s
own namespace that does *not* start with an underscore — which could accidentally
include things like the `logging` module itself, since `__init__.py` imported it.
`__all__` makes the exposed API explicit instead of accidental.

<a id='5'></a>
## 5. Import styles: absolute vs relative

- **Absolute import** — full path from the project root. Unambiguous, works anywhere.
- **Relative import** — path relative to the *current* module's position in its
  own package, using leading dots (`.` = same package, `..` = parent package).

In [23]:
shutil.rmtree(pkg)
pkg.mkdir()
(pkg / "models").mkdir()

(pkg / "config.py").write_text('''
DEFAULT_CURRENCY = "NGN"
''')

(pkg / "models" / "__init__.py").write_text("")
(pkg / "models" / "forest.py").write_text('''
# Absolute import: full path from the project root
from mypackage.config import DEFAULT_CURRENCY

# Relative import: ".." = go up one level to the parent package (mypackage)
from ..config import DEFAULT_CURRENCY as DEFAULT_CURRENCY_REL

def describe():
    return f"absolute={DEFAULT_CURRENCY}, relative={DEFAULT_CURRENCY_REL}"
''')

(pkg / "__init__.py").write_text("")

for name in list(sys.modules):
    if name == "mypackage" or name.startswith("mypackage."):
        del sys.modules[name]

from mypackage.models import forest
print(forest.describe())


absolute=NGN, relative=NGN


Both styles resolved the same value here — the point is **where each is
appropriate**: absolute imports are clearer when crossing package boundaries or
when a line might be copy-pasted elsewhere; relative imports are convenient for
referring to close siblings within the same package that might get moved/renamed
together.

<a id='6'></a>
## 6. Relative imports: `import ..x` is invalid, `from .. import x` is valid

This is a real Python syntax rule, not a style choice.

```python
import ..config          # SyntaxError — always, no exceptions
from ..config import DEFAULT_CURRENCY   # valid
from .. import config    # also valid
```

**Why:** the `import` statement's grammar only accepts a *dotted name*
(`import a.b.c`) — it does not accept a name that *starts* with a dot. Leading
dots are only valid in the `from <dotted-or-relative-name> import <names>` form,
because the `from` clause has special grammar that explicitly allows a prefix of
dots to mean "relative to here." `import` by itself has no such grammar — it was
never designed to express relative position, only an absolute dotted path.

So the rule isn't "relative imports are discouraged with `import`" — it's that
**`import ..x` is not valid Python syntax at all**, under any circumstance.

In [24]:
try:
    exec("import ..config")
except SyntaxError as e:
    print("SyntaxError as expected:", e)


SyntaxError as expected: invalid syntax (<string>, line 1)


<a id='7'></a>
## 7. `__name__` and the `__main__` guard

Every module has a `__name__` attribute.
- Run directly → `__name__ == "__main__"`.
- Imported → `__name__` is set to the module's own name.

This lets one file serve as both a script and an importable module.

In [25]:
Path("guard_demo.py").write_text('''
print("top-level code runs no matter what; __name__ =", __name__)

def useful():
    return "importable"

if __name__ == "__main__":
    print("This only runs on DIRECT execution")
    print(useful())
''')

print("--- running directly via subprocess ---")
import subprocess
result = subprocess.run([sys.executable, "guard_demo.py"], capture_output=True, text=True)
print(result.stdout)

print("--- importing instead ---")
import guard_demo
print("useful() still callable:", guard_demo.useful())


--- running directly via subprocess ---
top-level code runs no matter what; __name__ = __main__
This only runs on DIRECT execution
importable

--- importing instead ---
top-level code runs no matter what; __name__ = guard_demo
useful() still callable: importable


<a id='8'></a>
## 8. Running a package: what enables it, and why

### What enables `python -m some_package`
A package becomes runnable with `python -m <package>` when it contains a file
named **`__main__.py`**. When you run `python -m package_name`, Python:
1. Finds `package_name` on `sys.path` (same search as a normal import).
2. Imports it as a package (running `__init__.py`).
3. Then specifically looks for and executes `package_name/__main__.py` as
   the program's entry point.

### Why a developer needs this
- **CLI tools**: turns a library into something runnable from the terminal,
  e.g. `python -m pip`, `python -m http.server`, `python -m venv myenv`.
- **Distribution-friendly**: users don't need to know an internal file path
  (`python package/cli/commands.py`) — just the package name.
- **Consistent entry point**: works the same regardless of the current working
  directory, since `-m` resolves through the normal import system, not a
  relative file path.

In [26]:
runpkg = Path("runnable_pkg")
runpkg.mkdir()
(runpkg / "__init__.py").write_text("__version__ = '0.1'")
(runpkg / "core.py").write_text('''
def greet(name):
    return f"Hello, {name}!"
''')
(runpkg / "__main__.py").write_text('''
# Entry point executed by: python -m runnable_pkg
import sys
from .core import greet

def main():
    name = sys.argv[1] if len(sys.argv) > 1 else "World"
    print(greet(name))

if __name__ == "__main__":
    main()
''')

result = subprocess.run([sys.executable, "-m", "runnable_pkg", "Dubem"],
                         capture_output=True, text=True, cwd=".")
print(result.stdout, result.stderr)


Hello, Dubem!
 


<a id='9'></a>
## 9. `__main__.py` explained — and what an ML engineer stores in it

**What it is:** just a regular module, with one special role — it's the file
Python looks for and runs when the package is invoked with `-m`. It is *not*
required for a package to function as a library; it is only required if you
want the package to be *directly runnable* as a program.

**Is it necessary to have?** No. A pure library (e.g. a package only meant to
be `import`ed by other code, like `requests` or your own `mathutils`) doesn't
need one at all. You add it only when you want `python -m your_package` to do
something meaningful.

### As an ML engineer, what typically goes in `__main__.py`
Not model code itself — that belongs in proper modules (`train.py`,
`evaluate.py`, etc.). `__main__.py` is the **thin entry point** that:
- parses CLI arguments (`argparse`),
- wires together the real pipeline functions,
- and exits with a sensible return code.

Example: a package `ml_pipeline` where `__main__.py` lets you run
`python -m ml_pipeline train --epochs 5` or `python -m ml_pipeline evaluate`.

In [27]:
mlpkg = Path("ml_pipeline")
mlpkg.mkdir()
(mlpkg / "__init__.py").write_text("__version__ = '0.1'")
(mlpkg / "train.py").write_text('''
def train_model(epochs):
    return f"model trained for {epochs} epochs"
''')
(mlpkg / "evaluate.py").write_text('''
def evaluate_model():
    return "accuracy=0.93"
''')
(mlpkg / "__main__.py").write_text('''
# Thin CLI entry point. All real logic lives in train.py / evaluate.py.
import argparse
import sys

from .train import train_model
from .evaluate import evaluate_model

def build_parser():
    p = argparse.ArgumentParser(prog="ml_pipeline")
    sub = p.add_subparsers(dest="command", required=True)
    t = sub.add_parser("train")
    t.add_argument("--epochs", type=int, default=1)
    sub.add_parser("evaluate")
    return p

def main():
    args = build_parser().parse_args()
    if args.command == "train":
        print(train_model(args.epochs))
    elif args.command == "evaluate":
        print(evaluate_model())
    return 0

if __name__ == "__main__":
    sys.exit(main())
''')

result = subprocess.run([sys.executable, "-m", "ml_pipeline", "train", "--epochs", "5"],
                         capture_output=True, text=True)
print(result.stdout)
result = subprocess.run([sys.executable, "-m", "ml_pipeline", "evaluate"],
                         capture_output=True, text=True)
print(result.stdout)


model trained for 5 epochs

accuracy=0.93



<a id='10'></a>
## 10. `PATH` vs `sys.path`

These are **two different things** that are easy to conflate:

| | `PATH` | `sys.path` |
|---|---|---|
| What it is | An **operating-system** environment variable | A **Python interpreter** variable (a list) |
| Controls | Where your shell/OS looks for *executable programs* (like finding `python`, `git`, `node` when you type their name) | Where the Python *import system* looks for modules/packages to import |
| Type | A single string of OS-specific paths separated by `;` (Windows) or `:` (Linux/Mac) | A Python `list` of strings, inspectable and editable at runtime |
| Example use | Lets you type `python` instead of the full path to `python.exe` | Lets `import numpy` succeed by finding numpy's folder |

`sys.path` is seeded partly *from* environment variables (like `PYTHONPATH`) and
partly from the interpreter's own installation layout — but it is its own
separate list once Python starts, and you can freely modify it during a
program's execution (`sys.path.append(...)`), which has zero effect on the
OS-level `PATH`.

In [41]:
import sys
for p in sys.path[:5]:
    print(p)
print(len(sys.path))


print(f'\n{"="*80}\n')

import os
# Get the raw PATH string
system_path = os.environ.get("PATH")
for path in system_path.split(";")[:5]:
    print(path)
print(len(system_path))

c:\Users\ACER\miniconda3\envs\p2p-env\python314.zip
c:\Users\ACER\miniconda3\envs\p2p-env\DLLs
c:\Users\ACER\miniconda3\envs\p2p-env\Lib
c:\Users\ACER\miniconda3\envs\p2p-env

10


c:\Users\ACER\miniconda3\envs\p2p-env
C:\Windows\system32
C:\Windows
C:\Windows\System32\Wbem
C:\Windows\System32\WindowsPowerShell\v1.0\
1339


<a id='11'></a>
## 11. Scenario: package lives in Downloads, you're working from Documents

If `my_custom_package/` is sitting in `C:\Users\ACER\Downloads` but your
current working directory (and script) is in `C:\Users\ACER\Documents`,
a plain `import my_custom_package` fails — Python never looked there.

**Fix: add that folder to `sys.path` before importing.**

In [42]:
# Simulating the scenario: a package in one folder, working from another
downloads_sim = Path.cwd().parent / "nb_workspace_downloads_sim"
if downloads_sim.exists():
    shutil.rmtree(downloads_sim)
downloads_sim.mkdir()
custom_pkg = downloads_sim / "my_custom_package"
custom_pkg.mkdir()
(custom_pkg / "__init__.py").write_text("GREETING = 'hi from Downloads'")

# Without adding the path, this fails:
for name in list(sys.modules):
    if name == "my_custom_package":
        del sys.modules[name]
try:
    import my_custom_package
except ModuleNotFoundError as e:
    print("Fails first:", e)

# Fix: insert the parent folder containing the package
sys.path.insert(0, str(downloads_sim))
import my_custom_package
print("Works now:", my_custom_package.GREETING)


Fails first: No module named 'my_custom_package'
Works now: hi from Downloads


**Three ways to do this in practice** (not just inside a notebook cell):

1. **Temporary, inside your script** (what we just did):
   ```python
   import sys
   sys.path.insert(0, r"C:\Users\ACER\Downloads")
   import my_custom_package
   ```
2. **Session-wide, via the `PYTHONPATH` environment variable** (Windows cmd):
   ```
   set PYTHONPATH=C:\Users\ACER\Downloads;%PYTHONPATH%
   ```
   This makes every Python process in that terminal session see the extra path,
   without editing any code.
3. **Permanent/proper fix**: install the package itself, e.g. `pip install -e .`
   from inside the Downloads folder (if it has a `pyproject.toml`), so it lands
   in `site-packages` and is found automatically from anywhere — no path hacks
   needed at all. This is the preferred long-term solution.

<a id='12'></a>
## 12. `sys.modules` and the lifecycle of an imported module

`sys.modules` is a **dictionary cache**: `{module_name: module_object}`, held in
memory for the life of the Python process.

### The lifecycle, step by step
1. You write `import foo` (anywhere, any number of times).
2. Python checks `sys.modules["foo"]` first.
   - **Found?** → return that cached object immediately. The file is *not*
     re-read or re-executed. This is why importing something 100 times across
     a program only runs its top-level code once.
   - **Not found?** → continue to step 3.
3. Python's finders search `sys.path` for a matching file/package.
4. The matching file's code is executed **top to bottom, once**, building up
   a namespace (its functions, classes, variables).
5. That namespace becomes a module object, which is:
   - **cached** in `sys.modules["foo"]`, and
   - **bound** to the name `foo` in whatever scope did the importing.
6. Every subsequent `import foo` anywhere in the program — different file,
   different function — hits step 2's cache and skips straight to returning
   the same object.

### "Memory lifecycle" — when does it go away?
Unlike a variable that can be garbage-collected once unused, an entry in
`sys.modules` normally **lives for the entire life of the process**. Python does
not automatically evict modules just because nothing currently references them.
It's only cleared when:
- the process exits, or
- you explicitly `del sys.modules["foo"]` (rare, mostly used in testing/reloading
  scenarios — exactly what we did above before each re-import demo!), or
- you use `importlib.reload(foo)`, which re-executes the file's code but
  keeps the same module object in the cache (updating it in place) rather
  than creating a fresh one.

In [43]:
import math_utils  # already imported earlier in this notebook

print("cached object:", sys.modules["math_utils"])
print("is the same object as the imported name:", sys.modules["math_utils"] is math_utils)

# Proof that re-importing does NOT re-run the module's top-level code:
Path("counter_mod.py").write_text('''
print("counter_mod is executing its top-level code")
count = 0
''')
import counter_mod   # prints the message once
import counter_mod   # no print this time -- cache hit
import counter_mod   # still no print

import importlib
importlib.reload(counter_mod)  # forces re-execution, prints the message again


cached object: <module 'math_utils' from 'c:\\Users\\ACER\\OneDrive\\Documents\\Repositories\\my_learning\\AI_ML\\programming_math\\python\\nb_workspace\\math_utils.py'>
is the same object as the imported name: True
counter_mod is executing its top-level code
counter_mod is executing its top-level code


<module 'counter_mod' from 'c:\\Users\\ACER\\OneDrive\\Documents\\Repositories\\my_learning\\AI_ML\\programming_math\\python\\nb_workspace\\counter_mod.py'>

<a id='13'></a>
## 13. Namespace packages — folder with no `__init__.py`

### What happens without `__init__.py`
Python 3.3+ allows a folder with no `__init__.py` to still be imported as a
package — a **namespace package**. But it behaves differently from a real one.

In [44]:
ns_pkg = Path("test_package")
ns_pkg.mkdir()
(ns_pkg / "module_a.py").write_text("def hello(): return 'hi from module_a'")

for name in list(sys.modules):
    if name == "test_package" or name.startswith("test_package."):
        del sys.modules[name]

import test_package
print("__file__ :", test_package.__file__)
print("__path__ :", test_package.__path__)
print("type of __path__:", type(test_package.__path__))


__file__ : None
__path__ : _NamespacePath(['c:\\Users\\ACER\\OneDrive\\Documents\\Repositories\\my_learning\\AI_ML\\programming_math\\python\\nb_workspace\\test_package'])
type of __path__: <class '_frozen_importlib_external._NamespacePath'>


**`my_package.__file__` is `None` for a namespace package — why:**
`__file__` normally points to the actual Python file Python *executed* to build
the module/package object. A namespace package has no `__init__.py`, so there is
no file for Python to execute — there's nothing for `__file__` to meaningfully
point at, so Python leaves it unset (`None`).

**`__path__` is a `_NamespacePath`, not a plain `list`**, because a namespace
package can legitimately be *assembled from multiple directories* across
`sys.path` that happen to share the same folder name (this is how multi-distribution
namespaces like `google.cloud.*` work in the wild). A plain `list` would be a
frozen snapshot; `_NamespacePath` can reflect directories discovered dynamically.

### Now add `__init__.py` and compare

In [45]:
(ns_pkg / "__init__.py").write_text("")

for name in list(sys.modules):
    if name == "test_package" or name.startswith("test_package."):
        del sys.modules[name]

import test_package
print("__file__ :", test_package.__file__)
print("__path__ :", test_package.__path__)
print("type of __path__:", type(test_package.__path__))


__file__ : c:\Users\ACER\OneDrive\Documents\Repositories\my_learning\AI_ML\programming_math\python\nb_workspace\test_package\__init__.py
__path__ : ['c:\\Users\\ACER\\OneDrive\\Documents\\Repositories\\my_learning\\AI_ML\\programming_math\\python\\nb_workspace\\test_package']
type of __path__: <class 'list'>


Now `__file__` points at `__init__.py` (the file that *was* executed), and
`__path__` is a plain `list` — because the package is now anchored to exactly
one directory, with no ambiguity left to resolve dynamically.

**The "split across two folders" risk** (why this is usually unwanted): if two
*different* directories on `sys.path` both contain a same-named, `__init__.py`-less
folder, Python silently merges them into one namespace package rather than
raising a conflict error — which can mask bugs (e.g. a stray leftover folder
accidentally shadowing part of a real installed package).

<a id='14'></a>
## 14. Viewing your namespace cleanly (checking for pollution)

After a `from module import *`, you may want to check what actually landed in
your namespace. A few clean ways to do this:

- **`dir()`** with no arguments — lists all names in the *current* local/global
  scope, sorted alphabetically. Filter out dunders for a clean view.
- **`globals()`** — returns the actual dict backing the global namespace; lets
  you inspect *values*, not just names.
- **`vars()`** — same as `globals()` when called with no arguments at module level.
- **`locals()`** — the local namespace inside a function.

**Best practice for checking pollution specifically:** snapshot `dir()` *before*
a star-import, then diff it against `dir()` *after* — that isolates exactly
what the star-import added, with no manual guessing.

In [ ]:
before = set(dir())

from mypackage import *

after = set(dir())

print(before, after, sep=("\n"))
added_by_star_import = sorted(after - before)
print("Names added by `from mypackage import *`:")
print(added_by_star_import)


{'_i19', 'guard_demo', '__name__', '_i4', 'visible', '_i15', 'ns', 'exit', 'folder', '__', 'Out', '_i14', 'after', '_i46', '_i3', 'custom_pkg', '_i5', 'test_package', '_i48', 'p', '_i13', '_iii', '_i43', '_i36', 'WORKDIR', '_i23', 'downloads_sim', 'subprocess', '__builtin__', '_i44', '_i17', 'shutil', '__builtins__', 'math_utils', '_i9', 'quit', 'before', '_i37', '_i39', '_i7', '_i6', '_i30', '_i41', '_i16', '_i29', '_43', '__spec__', 'name', 'counter_mod', '_', '_i49', '_i31', 'os', '_i45', 'mlpkg', 'result', 'pkg', '_i28', '_i8', '_i2', '_i47', '_oh', 'config', 'system_path', '_i38', '_dh', '__vsc_ipynb_file__', 'open', 'sys', '___', '_i42', 'mypackage', '_i12', 'ns_pkg', '_i26', '_i33', '_i34', '_i27', '__doc__', 'Path', '_i21', '__package__', '_i24', 'key', 'importlib', '_i32', 'exposed', 'get_ipython', '_ii', 'my_custom_package', '_i22', '_i10', '_i', '_i35', 'added_by_star_import', 'runpkg', '_i40', '_i25', 'forest', '_i18', '_i20', '_i11', 'path', '_i1', '_ih', 'models', 'In', '

In [17]:
# Filtering dunders out of a dir() listing for a "clean" view
clean_names = [n for n in dir() if not (n.startswith('__') and n.endswith('__'))]
print(clean_names[:15], "... (truncated)")


['In', 'Out', 'Path', 'WORKDIR', '_', '_13', '_dh', '_i', '_i1', '_i10', '_i11', '_i12', '_i13', '_i14', '_i15'] ... (truncated)


<a id='15'></a>
## 15. Dunders reference table

"Dunder" = **D**ouble **UNDER**score, e.g. `__init__`. Grouped by where they show up.

### Module/package-level dunders
| Dunder | What it holds / does |
|---|---|
| `__name__` | The module's name; `"__main__"` if run directly |
| `__file__` | Path to the file that was executed (absent/`None` for namespace packages, built-ins) |
| `__path__` | List of directories backing a *package* (not set on plain modules) |
| `__package__` | The name of the package a module belongs to |
| `__doc__` | The module's (or function's/class's) docstring |
| `__all__` | List controlling what `from x import *` exposes |
| `__version__` | Convention-only (not built into Python) — a package's version string |
| `__dict__` | The namespace of a module/object/class, as an actual dict |
| `__loader__` / `__spec__` | Import-machinery internals describing how the module was loaded |

### Class/instance dunders (the "magic methods")
| Dunder | Triggered by | 
|---|---|
| `__init__` | Called automatically when an object is constructed, to set up instance state |
| `__repr__` | `repr(obj)` — unambiguous developer-facing string |
| `__str__` | `str(obj)` / `print(obj)` — readable user-facing string |
| `__eq__` | `obj == other` |
| `__lt__`, `__le__`, `__gt__`, `__ge__` | `<`, `<=`, `>`, `>=` comparisons |
| `__len__` | `len(obj)` |
| `__iter__` | starting a `for x in obj:` loop |
| `__next__` | each step of that loop (on an iterator) |
| `__getitem__` | `obj[key]` |
| `__setitem__` | `obj[key] = value` |
| `__contains__` | `x in obj` |
| `__call__` | `obj(...)` — makes an instance itself callable like a function |
| `__enter__` / `__exit__` | entering/leaving a `with obj:` block |
| `__post_init__` | (dataclasses only) runs right after the auto-generated `__init__` |
| `__class__` | the type of an instance |

Below: a single class exercising several of these at once, with output showing
exactly which dunder fired for which operation.

In [18]:
class Money:
    def __init__(self, amount):          # construction
        self.amount = amount

    def __repr__(self):                   # repr(obj)
        return f"Money({self.amount})"

    def __str__(self):                    # print(obj)
        return f"₦{self.amount:,.2f}"

    def __eq__(self, other):              # ==
        return isinstance(other, Money) and self.amount == other.amount

    def __lt__(self, other):              # <
        return self.amount < other.amount

    def __add__(self, other):             # +
        return Money(self.amount + other.amount)

    def __len__(self):                    # len()
        return int(self.amount)

    def __call__(self):                   # obj()
        return f"Calling Money() returns formatted: {self}"


m1 = Money(5000)
m2 = Money(7000)

print(repr(m1))          # -> __repr__
print(str(m1))            # -> __str__ (print() uses this)
print(m1 == Money(5000))  # -> __eq__
print(m1 < m2)             # -> __lt__
print(m1 + m2)              # -> __add__, then __str__ to display it
print(len(m1))                # -> __len__
print(m1())                    # -> __call__
print(m1.__class__)             # -> __class__
print(m1.__dict__)               # -> __dict__


Money(5000)
₦5,000.00
True
True
₦12,000.00
5000
Calling Money() returns formatted: ₦5,000.00
<class '__main__.Money'>
{'amount': 5000}


In [19]:
# __module__-level dunders and __post_init__ (dataclasses)
from dataclasses import dataclass

print("module __name__:", __name__ if '__name__' in dir() else math_utils.__name__)
print("math_utils.__doc__:", repr(math_utils.__doc__))

@dataclass
class Account:
    owner: str
    balance: float

    def __post_init__(self):               # fires right after auto __init__
        if self.balance < 0:
            raise ValueError("balance cannot start negative")
        print(f"__post_init__ ran: account for {self.owner} validated")

acc = Account("Dubem", 1000)


module __name__: __main__
math_utils.__doc__: None
__post_init__ ran: account for Dubem validated


<a id='16'></a>
## 16. `unittest` — how it works, and is it a standard library module?

**Yes — `unittest` is part of the Python standard library.** No installation
needed; `import unittest` always works on any standard Python install.

### How it works, conceptually
1. You write a class that inherits from `unittest.TestCase`.
2. Inside it, you write methods whose names **start with `test_`** — this
   naming convention is how `unittest` discovers which methods are tests.
3. Inside each test method, you use **assertion methods** (`self.assertEqual`,
   `self.assertTrue`, `self.assertRaises`, etc.) instead of plain `assert` —
   these give much more informative failure messages.
4. A **test runner** collects all `TestCase` subclasses and their `test_*`
   methods, runs each one in isolation, and reports pass/fail/error for each,
   plus a final summary.
5. Optional `setUp(self)` / `tearDown(self)` methods run automatically
   **before and after every single test method** in the class — useful for
   creating/cleaning up shared fixtures (e.g. a temp file) without repeating
   that code in every test.

### Running tests
- From Python: `unittest.main()` at the bottom of a test file.
- From the command line: `python -m unittest test_file.py`
- Auto-discovery across a project: `python -m unittest discover -s tests -v`
  (finds every `test_*.py` file under `tests/` automatically).

In [20]:
import unittest

class TestMoney(unittest.TestCase):

    def setUp(self):
        # Runs before EVERY test method below
        self.m1 = Money(100)
        self.m2 = Money(200)

    def test_addition(self):
        result = self.m1 + self.m2
        self.assertEqual(result.amount, 300)

    def test_equality(self):
        self.assertEqual(Money(50), Money(50))
        self.assertNotEqual(Money(50), Money(51))

    def test_comparison(self):
        self.assertTrue(self.m1 < self.m2)

    def test_negative_balance_raises(self):
        with self.assertRaises(ValueError):
            Account("Test", -5)

    def tearDown(self):
        # Runs after EVERY test method -- cleanup goes here
        pass

# Running inside a notebook: unittest.main() would try to exit the kernel,
# so we use a runner directly instead, which is the equivalent of
# `python -m unittest` from a terminal.
suite = unittest.TestLoader().loadTestsFromTestCase(TestMoney)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)


test_addition (__main__.TestMoney.test_addition) ... 

ok


test_comparison (__main__.TestMoney.test_comparison) ... 

ok


test_equality (__main__.TestMoney.test_equality) ... 

ok


test_negative_balance_raises (__main__.TestMoney.test_negative_balance_raises) ... 

ok


----------------------------------------------------------------------
Ran 4 tests in 0.005s

OK


<unittest.runner.TextTestResult run=4 errors=0 failures=0>

<a id='17'></a>
## 17. Summary cheat-sheet

| Concept | One-line takeaway |
|---|---|
| Module | A single `.py` file; its top-level code runs once, on first import |
| Package | A directory of modules; traditionally needs `__init__.py` |
| `__init__.py` | Runs once on first import; sets init code, `__all__`, re-exports |
| Namespace package | No `__init__.py`; `__file__` is `None`; `__path__` is `_NamespacePath`; can span folders |
| Absolute import | Full dotted path from project root; unambiguous |
| Relative import | `.`/`..` prefix, only valid with `from ... import ...`, never plain `import` |
| `__name__` | `"__main__"` when run directly, module name when imported |
| `__main__.py` | Lets `python -m package` work; optional — only needed for runnable packages |
| `sys.path` | Python's own list of directories searched for imports; editable at runtime |
| `PATH` | OS-level variable for finding *executables*, unrelated to Python imports |
| `sys.modules` | Cache dict of already-imported modules; re-import = cache hit, no re-execution |
| `unittest` | Standard library testing framework; `TestCase` + `test_*` methods + assertions |

---
*End of notebook.*
